In [0]:
%py

# # Widgets para testing interactivo
# dbutils.widgets.dropdown("mode", "historical", ["historical", "incremental", "full_refresh"])
# dbutils.widgets.text("max_pages", "3")  # Solo 3 páginas = 3,000 modelos para prueba rápida


In [0]:
%py
from datetime import datetime, timezone
import csv, json, os, time, uuid
from urllib.parse import parse_qs, urlparse

import requests

def get_param(name, default):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

CATALOG    = get_param("catalog", "pf")
MODE       = get_param("mode", "incremental")       # historical | incremental | full_refresh
MAX_PAGES  = int(get_param("max_pages", "50"))      # paginas de 1000 para historical/full

HF_API     = "https://huggingface.co/api/models"
VOLUME_RAW = f"/Volumes/pf/landing/files"
LANDING_CTL = f"{CATALOG}.bronze.ingestion_control"

NOW   = datetime.now(timezone.utc)
ING_TS = NOW
ING_DAY = NOW.strftime('%Y-%m-%d')

print(f"modo={MODE} max_pages={MAX_PAGES} landing={VOLUME_RAW}")


In [0]:
%py

def get_watermark():
    """Ultimo watermark exitoso (spark.sql). None si no hay corridas previas."""
    try:
        rows = spark.sql(f"""
            SELECT watermark_after
            FROM {LANDING_CTL}
            WHERE status = 'SUCCESS'
            ORDER BY end_ts DESC
            LIMIT 1
        """).collect()
    except Exception:
        return None
    if not rows or rows[0].watermark_after is None:
        return None
    wm = rows[0].watermark_after
    if wm.tzinfo is None:
        wm = wm.replace(tzinfo=timezone.utc)          # spark devuelve naive UTC
    return wm

CSV_FIELDS = [
    "_id", "id", "modelId", "likes", "private", "downloads",
    "pipeline_tag", "library_name", "createdAt", "lastModified",
    "tags", "payload_json",
    "ingesta_run_id", "ingesta_mode", "page_no",
    "ingestion_ts", "ingestion_date",
]

def write_page(run_dir, run_id, page_no, items):
    """Escribe una pagina como CSV en el Volume (FUSE: /Volumes/...)."""
    path = f"{run_dir}/page_{page_no:05d}.csv"
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=CSV_FIELDS, extrasaction="ignore")
        writer.writeheader()
        for it in items:
            row = {
                "_id":            it.get("_id"),
                "id":             it.get("id"),
                "modelId":        it.get("modelId"),
                "likes":          it.get("likes"),
                "private":        it.get("private"),
                "downloads":      it.get("downloads"),
                "pipeline_tag":   it.get("pipeline_tag"),
                "library_name":   it.get("library_name"),
                "createdAt":      it.get("createdAt"),
                "lastModified":   it.get("lastModified"),
                "tags":           json.dumps(it.get("tags", []), ensure_ascii=False),
                "payload_json":   json.dumps(it, ensure_ascii=False),
                "ingesta_run_id": run_id,
                "ingesta_mode":   MODE,
                "page_no":        page_no,
                "ingestion_ts":   ING_TS.isoformat(),
                "ingestion_date": ING_DAY,
            }
            writer.writerow(row)
    return path

def parse_iso(s):
    """'2026-09-01T12:00:00.000Z' -> naive UTC datetime (robusto)."""
    if not s:
        return None
    s = s.replace("Z", "+00:00")
    try:
        return datetime.fromisoformat(s).astimezone(timezone.utc)
    except ValueError:
        return None

def get_page(cursor, sort, direction):
    """GET con reintentos/backoff ante 429/5xx.
    Devuelve (items, next_cursor): los modelos de esta pagina y el cursor de la
    siguiente (None si no hay mas paginas). El cursor se lee del header
    `Link: <...cursor=XXX>; rel="next"` de la respuesta."""
    params = {"limit": 1000, "full": True, "sort": sort, "direction": direction}
    if cursor:
        params["cursor"] = cursor
    for attempt in range(5):
        try:
            r = requests.get(HF_API, params=params, timeout=60)
        except requests.RequestException as exc:
            time.sleep(min(60, 5 * (attempt + 1)))
            continue
        if r.status_code == 200:
            next_cursor = None
            if "next" in r.links:                       # header Link: rel="next"
                qs = parse_qs(urlparse(r.links["next"]["url"]).query)
                next_cursor = qs.get("cursor", [None])[0]
            return r.json(), next_cursor
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(min(60, 5 * (attempt + 1)))
        else:
            r.raise_for_status()
    raise RuntimeError("Limite de reintentos excedido")
#####################################################################################


In [0]:
%py

run_id  = ING_TS.strftime('%Y%m%dT%H%M%SZ') + "_" + uuid.uuid4().hex[:6]
run_dir = f"{VOLUME_RAW}/run_id={run_id}"
os.makedirs(run_dir, exist_ok=True)
wm_before = get_watermark()

if MODE in ("historical", "full_refresh"):
    sort, direction = "downloads", "-1"
else:  # incremental: modelos nuevos + actualizados desde el watermark
    sort, direction = "lastModified", "-1"

page, n_pages, n_records = 1, 0, 0
cursor = None                                   # None = primera pagina
items_in_run = []                            # para calcular watermark_after
notes = []

if MODE in ("historical", "full_refresh"):
    max_loop_pages = MAX_PAGES
else:
    max_loop_pages = 200                     # cota de seguridad para incremental

while n_pages < max_loop_pages:
    data, next_cursor = get_page(cursor, sort, direction)
    if not data:
        break
    if MODE == "incremental":
        # Pagina ordenada desc por lastModified: si NINGUN modelo es >= watermark,
        # todas las siguientes son mas viejas -> detener.
        if wm_before is not None and all(
            (parse_iso(i.get("lastModified")) or datetime.min.replace(tzinfo=timezone.utc)) < wm_before
            for i in data
        ):
            notes.append(f"detenido en page {page}: todo anterior al watermark")
            break
    write_page(run_dir, run_id, page, data)
    items_in_run.extend(data)
    n_pages += 1
    n_records += len(data)
    print(f"page {page}: {len(data)} modelos (run_id={run_id})")
    page += 1
    if next_cursor is None:                  # la API no devuelve mas paginas
        notes.append(f"fin de paginacion en page {page - 1}")
        break
    cursor = next_cursor
    time.sleep(1)                                # rate-limit friendly

# watermark_after: max(lastModified) observado en esta corrida, o el timestamp de corrida.
# Se rastrea en el bucle para evitar depender de re-leer los CSV.
wm_after = ING_TS
for i in items_in_run:                      # items_in_run se acumula en el bucle
    lm = parse_iso(i.get("lastModified"))
    if lm and lm > wm_after:
        wm_after = lm

status = "SUCCESS" if n_records > 0 else "PARTIAL"
notes = notes or ["sin registros nuevos"]
########################################################################

In [0]:
%py

def ts_lit(dt):
    """Timestamp literal compat con Spark (sin T/Z): YYYY-MM-DD HH:MM:SS."""
    if dt is None:
        return "NULL"
    return f"TIMESTAMP '{dt.astimezone(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}'"

spark.sql(f"""
    INSERT INTO {LANDING_CTL} (run_id, mode, start_ts, end_ts,
                               watermark_before, watermark_after,
                               n_pages, n_records, status, notes)
    SELECT  '{run_id}', '{MODE}',
            {ts_lit(ING_TS)},
            {ts_lit(datetime.now(timezone.utc))},
            {ts_lit(wm_before)},
            {ts_lit(wm_after)},
            {n_pages}, {n_records}, '{status}', '{';'.join(notes)}'
""")

print(f"Run {run_id}: pages={n_pages} records={n_records} status={status}")
print(f"CSV en: {run_dir}")
#########################################################################


In [0]:
%py

spark.sql(f"""
    SELECT run_id, mode, status, watermark_before, watermark_after, n_pages, n_records
    FROM {LANDING_CTL}
    ORDER BY end_ts DESC LIMIT 5
""").show()

spark.sql(f"SELECT COUNT(*) AS files FROM read_files('{VOLUME_RAW}/run_id={run_id}/*.csv', format => 'csv', header => true, multiLine => true)").show()
######################################################################